# toro-CM1 no vlab — Lhende Khola, 25–26/08/2026

Roda as rodadas do CM1 de `docs/TORO_CM1_SONDAGEM_REAL_ERA5.md` no Jupyter do vlab (`jupyter.vlab.ufsc.br`) e devolve os resultados pelo Google Drive.

**Como é o vlab** (aprendido na primeira instalação):

| Fato | Consequência |
| :--- | :--- |
| contêiner rodando como `root`, `HOME=/root` | **`/root` pode ser apagado ao reiniciar o servidor** → tudo fica em `/home/jovyan/privado` |
| sem Slurm (`sbatch` não existe) | o modelo roda **na própria sessão**, em segundo plano (`rodar_direto.sh`) |
| sem scp/rsync | pacote vem do **GitHub** (público) ou do **Drive**; resultados voltam pelo **Drive** |
| o rclone lê `/root/.config/rclone` por padrão | sempre `source ambiente.sh` (exporta `RCLONE_CONFIG` persistente) |

**Fluxo de cada sessão:** célula 1 (restaurar) → 2 (pacote) → 3 (compilar, só se preciso) → 5 a 7 (rodar e acompanhar) → 8 (devolver).
A célula 4 (Drive) e a 9 (Claude Code) são só da primeira vez ou depois de um reinício que apagou `/root`.

Pasta de trabalho: `/home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende`

## 1. Restaurar o ambiente (rode sempre primeiro)
Verifica e refaz o que se perde em `/root`: links do login do Claude Code, PATH, rclone e bibliotecas do `cm1.exe`. Mostra também a fila. Não instala nada sozinho.

Se a pasta ainda não existe (primeira vez), pule para a célula 2.

In [ ]:
%%bash -l
df -h /root /home/jovyan/privado 2>/dev/null | sed 's/^/   /'
if [ -d /root/mesoescala-montanhas ] && [ ! -d /home/jovyan/privado/mesoescala-montanhas ]; then mv /root/mesoescala-montanhas /home/jovyan/privado/mesoescala-montanhas && echo "movido de /root para /home/jovyan/privado/mesoescala-montanhas (recompile na celula 3)"; fi
if [ -f /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende/restaurar.sh ]; then cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && bash restaurar.sh; else echo "pacote ainda nao instalado: rode a celula 2"; fi


## 2. Baixar ou atualizar o pacote
Usa o **Drive** se o remoto `gdrive:` já estiver configurado; senão, o **GitHub**. Confere o SHA-256 e extrai **por cima**: o `cm1/run/cm1.exe` compilado, `execucoes/`, `logs/` e os resultados ficam intactos.

In [ ]:
%%bash -l
mkdir -p /home/jovyan/privado/mesoescala-montanhas && cd /home/jovyan/privado/mesoescala-montanhas
ORIGEM=github
if [ -f /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende/ambiente.sh ]; then source /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende/ambiente.sh; command -v rclone >/dev/null || export PATH=$ENVDIR/cm1pos/bin:$PATH
  rclone lsd gdrive:vlab_cache >/dev/null 2>&1 && ORIGEM=drive; fi
if [ $ORIGEM = drive ]; then
  rclone copy gdrive:vlab_cache/entrada/toro_cm1_lhende_vlab.tar.gz . 2>&1 | grep -v client_id || true
  rclone copy gdrive:vlab_cache/entrada/toro_cm1_lhende_vlab.tar.gz.sha256 . 2>&1 | grep -v client_id || true
else
  U=https://raw.githubusercontent.com/reinaldohaas/Himalaia_2026/refs/heads/claude/himalaia-enchentes-auditoria-fcfdb5/cluster
  wget -q -O toro_cm1_lhende_vlab.tar.gz $U/toro_cm1_lhende_vlab.tar.gz && wget -q -O toro_cm1_lhende_vlab.tar.gz.sha256 $U/toro_cm1_lhende_vlab.tar.gz.sha256
fi
echo "origem: $ORIGEM"
sha256sum -c toro_cm1_lhende_vlab.tar.gz.sha256 || { echo "SHA-256 NAO confere: nao extraido"; exit 1; }
tar xzf toro_cm1_lhende_vlab.tar.gz --overwrite
cd toro_cm1_lhende && sha256sum -c --quiet MANIFESTO.sha256 && echo "manifesto OK"


## 3. Compilar (só se preciso, ~2–5 min)
Compila se o `cm1.exe` não existe ou perdeu as bibliotecas (ambiente do compilador apagado). Se o gfortran não estiver no sistema, cria `/home/jovyan/privado/conda_envs/cm1`.

In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
[ -f cm1/run/cm1.env ] && { source cm1/run/cm1.env; export LD_LIBRARY_PATH=$ENV_PREFIX/lib:${LD_LIBRARY_PATH:-}; }
if [ -x cm1/run/cm1.exe ] && ! ldd cm1/run/cm1.exe | grep -q "not found"; then echo "cm1.exe ok, nao precisa compilar"
else NPROC=8 bash compilar.sh; fi


## 4. Google Drive (só na primeira vez, ou se a célula 1 acusar problema)
O login do Google precisa de navegador, então é feito **no seu Windows**:
1. PowerShell: `winget install Rclone.Rclone`, reabrir, `rclone config create gdrive drive scope=drive` (abre o navegador, entre e autorize), `rclone lsd gdrive:vlab_cache`.
2. Copie `%APPDATA%\rclone\rclone.conf` para `Downloads\` (a pasta AppData é oculta e não aparece no seletor de arquivos).
3. Aqui no Jupyter: painel de arquivos → `privado/` → **Upload** → `rclone.conf`.
4. Rode a célula abaixo. Depois apague a cópia em `Downloads\`.

O `rclone.conf` contém o token do seu Drive: não cole o conteúdo em lugar nenhum. O aviso `NOTICE … client_id is being retired` é inofensivo por enquanto; se o Drive parar de responder, crie um client_id próprio (https://rclone.org/drive/#making-your-own-client-id) e refaça o passo 1.

In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
bash gdrive.sh instalar
[ -f /home/jovyan/privado/rclone.conf ] && bash gdrive.sh importar || echo "(sem privado/rclone.conf novo para importar)"
bash gdrive.sh estado 2>&1 | grep -v client_id; exit ${PIPESTATUS[0]}


## 5. Recursos e teste curto
Mostra os núcleos e a memória reais da sessão e roda 120 s de modelo da `rad_12utc` (≈1 min). Rode a célula seguinte até aparecer `concluido`.

In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
bash rodar_direto.sh recursos
bash rodar_direto.sh -t rad_12utc


In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
bash rodar_direto.sh estado
tail -n 3 execucoes/rad_12utc_teste/execucao.info 2>/dev/null || echo "(teste ainda rodando: rode esta celula de novo em alguns minutos)"


## 6. Rodar
Se o teste terminou com `Program terminated normally`: as duas com radiação (`rad_06utc` é a principal, 21 h de modelo — várias horas de relógio). Para tudo: `todos`. Para fixar threads: `-j 16`.

**O modelo vive enquanto o seu servidor do Jupyter estiver ligado.** Não clique em *Stop My Server*. Fechar o notebook ou o Claude não interrompe.

In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
bash rodar_direto.sh rad


## 7. Acompanhar
Pode rodar quantas vezes quiser. Para interromper: `bash rodar_direto.sh parar`.

In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
bash rodar_direto.sh estado


## 8. Devolver ao seu computador (Drive)
`relatorio.sh --enviar` manda o estado das execuções (para verificação) a qualquer momento. Ao terminar as rodadas, a análise gera figuras e JSON e o pacote leve (poucos MB) vai para o Drive.

No Windows tudo aparece em `G:\Meu Drive\vlab_cache\saida\toro_cm1_lhende\`.

In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
bash relatorio.sh --enviar


In [ ]:
%%bash -l
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source ambiente.sh
bash pos/analisar.sh
bash coletar.sh --leve
bash gdrive.sh enviar 2>&1 | grep -v client_id; R=${PIPESTATUS[0]}
# saidas 3D completas (~11 GB, consome cota do Drive):  bash coletar.sh && bash gdrive.sh enviar --completo
exit $R


## 9. Claude Code no vlab (opcional; no **Terminal**, não aqui)
O `claude` é interativo: rode num Terminal do Jupyter (*File → New → Terminal*). A célula 1 já refaz os links do login em `/home/jovyan/privado/.claude_persist`; se o `claude` sumiu após um reinício, reinstale:

```bash
cd /home/jovyan/privado/mesoescala-montanhas/toro_cm1_lhende && source restaurar.sh
curl -fsSL https://claude.ai/install.sh | bash      # so se a celula 1 disser "NAO instalado"
claude                                              # na pergunta de confianca: seta para baixo -> "Yes, I trust this folder"
```

Ele lê o `CLAUDE.md` desta pasta (contexto e regras). No terminal, sempre `source restaurar.sh` antes de usar `rclone` à mão; e prefira `rclone lsd gdrive:` a `rclone ls gdrive:` (este lista o Drive inteiro).